In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers



In [45]:
# =========================
# 1. CONFIG
# =========================
TRAIN_PATH = "D:/post_kuliah/assessment/project-2/classification_dataset/train"
VAL_PATH   = "D:/post_kuliah/assessment/project-2/classification_dataset/valid"
TEST_PATH  = "D:/post_kuliah/assessment/project-2/classification_dataset/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

In [46]:
# =========================
# 2. LOAD DATASET
# =========================
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 2609 files belonging to 4 classes.
Found 745 files belonging to 4 classes.
Found 373 files belonging to 4 classes.
Classes: ['depan', 'miring_kanan', 'miring_kiri', 'nunduk']


In [47]:
# =========================
# 3. OPTIMIZE
# =========================
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [48]:
# =========================
# 4. MODEL (NO AUGMENTATION)
# =========================
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = tf.keras.applications.mobilenet_v3.preprocess_input(inputs)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = keras.Model(inputs, outputs)

In [49]:
# =========================
# 6. COMPILE
# =========================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [50]:
# =========================
# 7. TRAIN
# =========================
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)


Epoch 1/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 17s 154ms/step - accuracy: 0.7497 - loss: 0.6383 - val_accuracy: 0.9248 - val_loss: 0.2782
Epoch 2/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 11s 129ms/step - accuracy: 0.9279 - loss: 0.2539 - val_accuracy: 0.9477 - val_loss: 0.1735
Epoch 3/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 127ms/step - accuracy: 0.9598 - loss: 0.1561 - val_accuracy: 0.9544 - val_loss: 0.1324
Epoch 4/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 128ms/step - accuracy: 0.9644 - loss: 0.1261 - val_accuracy: 0.9678 - val_loss: 0.1102
Epoch 5/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 126ms/step - accuracy: 0.9728 - loss: 0.0962 - val_accuracy: 0.9664 - val_loss: 0.1046
Epoch 6/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 127ms/step - accuracy: 0.9801 - loss: 0.0780 - val_accuracy: 0.9732 - val_loss: 0.0760
Epoch 7/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 125ms/step - accuracy: 0.9843 - loss: 0.0628 - val_accuracy: 0.9745 - val_loss: 0.0675
Epoch 8/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 126ms/step - accuracy: 0.9877 - loss: 0.0521 - val_accu

In [51]:
# =========================
# 8. EVALUATE
# =========================
test_loss, test_acc = model.evaluate(test_ds)
print("Test accuracy:", test_acc)

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 104ms/step - accuracy: 0.9678 - loss: 0.0739
Test accuracy: 0.9678283929824829


In [52]:
# =========================
# 9. SAVE + TFLITE
# =========================
model.save("mobilenetv3_models.keras")

In [54]:
model = tf.keras.models.load_model("mobilenetv3_models.keras")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\felix\AppData\Local\Temp\tmp1ifvnwvz\assets


INFO:tensorflow:Assets written to: C:\Users\felix\AppData\Local\Temp\tmp1ifvnwvz\assets


Saved artifact at 'C:\Users\felix\AppData\Local\Temp\tmp1ifvnwvz'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_14')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  1629391048368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629391059280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629391110736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629391054704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629391056992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629391121648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629391119712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629391122880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629391118832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629391122176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  162